<a href="https://colab.research.google.com/github/OSGeoLabBp/tutorials/blob/master/hungarian/pontcloud/pointclouds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Pontfelhők feldolgozása

Ez az összeálltás a gita Műszaki térinformatika egyesület 2026. évi konferenciájának workshopjához készült. A tananyag a BME Általános és Felsőgeodézia Tanszék Geo4All laborjának GitHub oldaláról letölthetők.

## Bevezetés

### Pontfelhőkben tárolt adatok

A pontfelhőkben a pozíció mellett további adatok is tárolásra kerülhetnek. A LiDAR technológiával előállított pontfelhőkben sokszor  visszaverődés intenzitását is tárolják. A fotogrammetriai módszerrel előállított pontfelhőkben szín (RGB) értéket is tárolnak. Szín értékek a LiDAR pontfelhőkben is lehetnek, ha fényképek is készülnek a szkenneléssel párhuzamosan. A légi LiDAR felmérések esetén gyakran osztályozás adatot is találhatunk a pontfelhőben.

### Pontfelhők tárolási formátumai

A térinformatikához hasonlóan sokféle adatformátum használatos a pontfelhő állományok tárolására. Ezek között találhatók nyílt és zárt formátumok.

A nyílt formátumok esetén az állomány belső struktúrájának leírása bárki számára elérhető és ez alapján készíthet olyan programot, mely a formátumot olvasni és írni képes. Néhány, Magyarországon elterjedt nyílt pontfelhő adatformátum: LAS/LAZ, e57, XYZ, PLY, PCD

A zárt formátumok egyes kereskedelmi szotverekhez kötődnek, azokat csak a megfelelő szoftvercsomag megvásárlása esetén tudjuk kezelni. Ilyen formátumok például az RCS és a RCP.

Az egyes tárolási formátumok közötti választásnál mérlegelni kell a metaadatok (intenzitás, szín, osztályozás) megőrzését, a fájlméretet/tömöríthetőséget illetve a szoftver kompatibilitást.

Az utóbbi években egyre több program támogatja a felhőre optimalizált (copc - cloud optimized point cloud) állományokat.

### Pontfelhők előállítási technológiái

A pontfelhők előállításának két fő technológiája létezik a lézeres távmérés (LiDAR) és a fotogrammetria (SfM). A LiDAR technológia több részterületre bontható TLS, ALS, SLAM, stb.

### Nyílt forráskódú programok pontfelhők kezelésére

*   CloudCompare - grafikus felhasználói felület és programozási
*   PDAL - programkönyvtár és parancssori eszközök
*   PCL - programkönyvtár
*   Open3D - programkönyvtár Python programokhoz
*   Potree - internetes megjelenítés



### Szükséges programkönyvtárak és adatok teltöltése

A példák során a Python programnyelvet és az Open3D könyvtárat használjuk.

In [ ]:
!pip uninstall -q -y ipywidgets
!pip install -q open3d
!pip install -q cloth-simulation-filter
!pip install -q plotly

A feladatok során néhány mintaállományt használunk

In [ ]:
!wget -q https://github.com/OSGeoLabBp/tutorials/raw/refs/heads/master/hungarian/pointcloud/data/GITA_Test1_Off-ground_points.las
!wget -q https://github.com/OSGeoLabBp/tutorials/raw/refs/heads/master/hungarian/pointcloud/data/minta1.ply

In [ ]:
!wget -q https://raw.githubusercontent.com/zsiki/pygez/refs/heads/main/src/ransac.py
!wget -q https://raw.githubusercontent.com/zsiki/pygez/refs/heads/main/src/regression.py

## Nyers pontfelhők előkészítése

A nyers pontfelhők jellemzően zajjal terheltek. Olyan pontok jelenhetnek benne,melyek a valóságban nincsenek illetve a pontok sűrűsége túl nagy az adott feladatunkhoz. Mindkét esetben a pontfelhőben lévő pontok közül bizonyosakat eltávolítunk. A döntési kritériumban van különbség, ami alapján egy pontot megőrzünk vagy kihagyunk.

### Zajszűrési módszerek

A zajszűrésnél használt fontosabb módszerek:

* statisztikai módszerek (Statistical Outlier Removal, Radius Outlier Removal)
* geometriai módszerek (pl. RANSAC)
* PCA alapú módszerek
* mélytanulás alapú módszerek

In [ ]:
import copy
from math import atan2, hypot, pi
import numpy as np
import shapely
import open3d as o3d
from matplotlib import pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import CSF
from scipy.interpolate import griddata, RegularGridInterpolator
from ransac import Ransac
from regression import LinearReg, CircleReg

Egy segédfüggvény a pontfelhők kisebb felbontású megjelenítéséhez.

In [ ]:
def pc_show(pc, width=800, height=600, voxel_size=0.1, extra=None):
    """ Pontfelhő megjelenítése csökkentett felbontással
        az extra paraméter xyz tömbök listája, a pontokat egyenessel köti össze
    """
    if voxel_size > 0:
        pc1 = pc.voxel_down_sample(voxel_size=voxel_size)
    else:
        pc1 = copy.deepcopy(pc)
    pts = np.asarray(pc1.points)
    cols = np.asarray(pc1.colors)

    # megjelenítés
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode='markers', marker=dict(size=1, color=cols,))
    )
    if extra is not None:
        for l in extra:
            fig.add_trace(go.Scatter3d(x=l[:,0], y=l[:,1], z=l[:,2], mode='lines', line=dict(color='red', width=12)))
    # testreszabás
    fig.update_layout(width=width, height=height,
        scene=dict(aspectmode='data', xaxis=dict(visible=False),
                   yaxis=dict(visible=False), zaxis=dict(visible=False)),
        margin=dict(l=0, r=0, b=0, t=0)
    )
    fig.show()

Egy segédfüggvény két pontfelhő részlet megjelenítéséhez.

In [ ]:
def pc2_show(part_pc1, part_pc2, title1, title2, width=800, height=400):
    """ Két pontfelhő megjelenítése egymás mellett"""
    pts1 = np.asarray(part_pc1.points)
    col1 = np.asarray(part_pc1.colors)
    pts2 = np.asarray(part_pc2.points)
    col2 = np.asarray(part_pc2.colors)
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=(title1, title2)
    )
    fig.add_trace(go.Scatter3d(
        x=pts1[:, 0], y=pts1[:, 1], z=pts1[:, 2],
        mode='markers',
        marker=dict(size=1, color=col1)
        ),
        row=1, col=1
    )
    fig.add_trace(go.Scatter3d(
        x=pts2[:, 0], y=pts2[:, 1], z=pts2[:, 2],
        mode='markers',
        marker=dict(size=1, color=col2)
        ),
        row=1, col=2
    )
    fig.update_layout(height=height, width=width,
        scene=dict(aspectmode='data', xaxis=dict(visible=False),
            yaxis=dict(visible=False), zaxis=dict(visible=False)),
        scene2=dict(aspectmode='data', xaxis=dict(visible=False),
            yaxis=dict(visible=False), zaxis=dict(visible=False)),
        margin=dict(l=0, r=0, b=0, t=0),
        showlegend=False
    )
    fig.show()

A minta állomány:

In [ ]:
pc = o3d.io.read_point_cloud('minta1.ply')
pc_show(pc)
bbox = o3d.geometry.AxisAlignedBoundingBox(min_bound=(550936, 182422, 285), max_bound=(550943, 182427, 299))
# eredeti részlet
part_pc = pc.crop(bbox)

### Statistical Outlier Removal (SOR)

A zaj pontok megtalálása statisztikai alapon. Számítsuk ki minden egyes pontra a **k** legközelebbi szomszéd átlagos távolságát.

$d_i = \frac {1} {k} \sum_{j=1}^k || p_i - p_{ij} ||$

Számítsuk ki ezen távolságok átlagát és szórását

$\mu = \frac 1 {n} \sum_{i=1}^n d_i$

$\sigma = \sqrt \frac {\sum_{i=1}^n (d_i - \mu)^2} {n-1}$

Vizsgáljuk meg minden pontra, hogy a "**k**" legközelebbi szomszéd átlagos távolsága nagyobb-e mint a globális átlag szórással megnövelt értéke.

$d_i > \mu + \alpha \cdot \sigma$

$\alpha$ általában 1-3 közé eső konstans.

Egyenletes pontsűrűségű pontfelhőkre adja a legjobb eredményt.

In [ ]:
NB_NEIGHBORS = 25   # szomszédos pontok száma 10 - 50 között
STD_RATIO = 3       # alfa 1-3 között
sor_pc, _ = pc.remove_statistical_outlier(nb_neighbors=NB_NEIGHBORS, std_ratio=STD_RATIO)
o3d.io.write_point_cloud('minta1_sor.ply', sor_pc)
print(f"{len(pc.points)-len(sor_pc.points)} pont eltávolítva a {len(pc.points)} közül, {100-len(sor_pc.points)/len(pc.points)*100:.1f}%.")

Egy részlet az eredeti és a szűrt pontfelhőből

In [ ]:
# szűrt részlet
sor_pc_part = sor_pc.crop(bbox)
pc2_show(part_pc, sor_pc_part, 'Szűrés előtt', 'Szűrés után')

### Radius Outlier Removal

A kevés közeli szomszéddal rendelkező pontokat tekintjük zajnak. Minden egyes pontra keressük meg egy "**R**" sugarú környezetébe eső pontokat, ha ezek száma egy korlátnál kevesebb, akkor eltávolítjuk a pontot.

In [ ]:
RADIUS = 0.3
LIMIT = 50
ror_pc, _ = pc.remove_radius_outlier(nb_points=LIMIT, radius=RADIUS)
o3d.io.write_point_cloud('minta1_ror.ply', ror_pc)
print(f"{len(pc.points)-len(ror_pc.points)} pont eltávolítva a {len(pc.points)} közül, {100-len(ror_pc.points)/len(pc.points)*100:.1f}%.")

In [ ]:
# szűrt részlet
ror_pc_part = ror_pc.crop(bbox)
pc2_show(part_pc, ror_pc_part, 'Szűrés előtt', 'Szűrés után')

### Zajszűrő

Egy pont környezetébe eső pontokra illesszünk egy síkot. A pontnak a síktól mért távolságára adott korlát alapján döntünk, hogy zajnak tekintjük-e a pontot.
Ez a szűrő sík lapokkal határolt objektumok esetén, például városias környezetben használható legjobban.

In [ ]:
# ez a szűrő nincs az Open3D könyvtárban, saját implementáció
kdtree = o3d.geometry.KDTreeFlann(pc)   # szomszédok kereséséhez
k = 20  # szomszédok száma
DIST_LIMIT = 0.1
pc_filtered = []
for i in range(len(pc.points)):
    p = np.asarray(pc.points)[i]    # vizsgált pont
    [count, idx, dists] = kdtree.search_knn_vector_3d(pc.points[i], k) # legközelebbi k pont
    if count < k:
        print(f"{i} nincs elég szomszéd")
    r = LinearReg(np.asarray(pc.points)[idx[1:], :])
    plane = r.lkn_reg()
    d = np.abs(np.dot(plane[:3], p)) + plane[3]
    if d > DIST_LIMIT:
        pc_filtered.append(i)
nf_pc = pc.select_by_index(pc_filtered, invert=True)
print(f"{len(pc.points)-len(nf_pc.points)} pont eltávolítva a {len(pc.points)} közül, {100-len(nf_pc.points)/len(pc.points)*100:.1f}%.")

In [ ]:
# szűrt részlet
nf_pc_part = nf_pc.crop(bbox)
pc2_show(part_pc, nf_pc_part, 'Szűrés előtt', 'Szűrés után')

### Pontfelhők ritkitása

A pontfelhő ritkítására léteznek nagyon egyszerű módszerek. Például minden **n.** pont eltávolítása vagy véletlenszerűen válaszott pontok eltávolítása. Ezeknél célszerűbb megoldás a szomszédos pontok közötti minimális távolsággal vagy pontsűrűség alapján történő szűrés.

#### Voxel alapú ritkítás

Osszuk fel a pontfelhő pontjait a minimális távolság méretű kis kockákra (voxelekre). Minden voxelen belül csak egy pontot őrizzünk meg, ha esnek pontok a voxelbe.

In [ ]:
VOXEL_SIZE = 0.1  # minimális távolság a pontok között
vox_pc = pc.voxel_down_sample(voxel_size=VOXEL_SIZE)
o3d.io.write_point_cloud('minta1_vox.ply', vox_pc)
print(f"{len(pc.points)-len(vox_pc.points)} pont eltávolítva a {len(pc.points)} közül, {100-len(vox_pc.points)/len(pc.points)*100:.1f}%.")

In [ ]:
# szűrt részlet
vox_pc_part = vox_pc.crop(bbox)
pc2_show(part_pc, vox_pc_part, 'Ritkítás előtt', 'Ritkítás után')

#### Nyolcas fa alapú ritkítás

A nyolcas fa egy 3D térbeli indexelési módszer is, melyet most a pontfelhő ritkítása során használjuk fel. A nyolcas fa kialakítása során a pontok befoglaló kockájából indulunk ki. A kocka éleinek felezésével nyolc kis kockára bonjuk, addig folytatjuk ezt a felbontást, amíg egy adott felbontási mélységet elérünk vagy az egyes a kisebb kockákba eső pontok száma egy korlátnál kisebb.

![nyolcasfa](https://upload.wikimedia.org/wikipedia/commons/thumb/2/20/Octree2.svg/250px-Octree2.svg.png)

Forrás: https://en.wikipedia.org/wiki/Octree

In [ ]:
# nyolcas fa
MAX_DEPTH = 10   # fa maximális mélysége a terjedelem és felbontás alapján
octree = o3d.geometry.Octree(max_depth=MAX_DEPTH)   # nyolcas fa étrehozása
octree.convert_from_point_cloud(pc)
down_points = []    # lista a ritkított pontok tárolásához
down_cols = []

# segédfüggvény a nyolcas fa bejárásához
def traverse(node, node_info):
    if isinstance(node, o3d.geometry.OctreeLeafNode):   # a fa levele?
        if len(node.indices) > 0:
            pts = np.asarray(pc.points)[node.indices]
            cols = np.asarray(pc.colors)[node.indices]
            centroid = pts.mean(axis=0)
            cen_col = cols.mean(axis=0)
            down_points.append(centroid)
            down_cols.append(cen_col)
    return False  # continue traversal

octree.traverse(traverse)   # nyolcas fa bejárása
# pontfelhő a szűrt pontokból
oct_pc = o3d.geometry.PointCloud()
oct_pc.points = o3d.utility.Vector3dVector(np.array(down_points))
oct_pc.colors = o3d.utility.Vector3dVector(np.array(down_cols))
# mentés fájlba
o3d.io.write_point_cloud("minta1_oct.ply", oct_pc)
print(f"{len(pc.points)-len(oct_pc.points)} pont eltávolítva a {len(pc.points)} közül, {100-len(oct_pc.points)/len(pc.points)*100:.1f}%")

In [ ]:
# szűrt részlet
oct_pc_part = oct_pc.crop(bbox)
pc2_show(part_pc, oct_pc_part, 'Ritkítás előtt', 'Ritkítás után')

## Pontfelhők szegmentálása

A pontfelhők előállítása során jellemzően egy strukturálatlan adatállomány áll elő. A feldolgozás során a pontfelhő szegmentálása egy fontos feladat. A talaj, növényzet, épület, stb. pontok elkülönítésére sokféle módszert dolgoztak ki.

### Talajpontok elkülönítése

A talajpontok elkülönítésére a Cloth Simulation Filter (CSF) egy elterjedt módszer. Az eljárás alapgondolata, hogy tükrözzük a pontfelhőt az XY síkra. Helyezzünk egy lepedőt a pontokra, mely a gravitáció következtében rásimul a pontokra, de az anyag "merevsége" miatt áthidalja a "réseket". A lepedő közelében található pontokat talajpontoknak minősítjük.

![CSF](https://www.researchgate.net/profile/Ilya-Afanasyev-3/publication/332034911/figure/fig2/AS:741272688017414@1553744589946/The-illustration-of-the-Cloth-Simulation-Filtering-CSF-algorithm-The-original-point_W640.jpg)

Forrás: https://www.researchgate.net/publication/332034911_Ground_Profile_Recovery_from_Aerial_3D_LiDAR-based_Maps/figures?lo=1

In [ ]:
pc_xyz = np.asarray(pc.points)
csf = CSF.CSF()
csf.params.cloth_resolution = 0.5   # rácsméret a szimulációhoz
csf.params.rigidness = 3            # 1 hegyvidék, 2 complex táj, 3 sima terület magas épületekkel
csf.params.class_threshold = 0.5    # távolságlimit a lepedőtől
csf.params.iterations = 500
csf.params.bSloopSmooth = True      # utólagos simítás

csf.setPointCloud(pc_xyz)
ground = CSF.VecInt()
non_ground = CSF.VecInt()
csf.do_filtering(ground, non_ground, exportCloth=False)
pc_ground = pc.select_by_index(ground)
pc_non_ground = pc.select_by_index(non_ground)
o3d.io.write_point_cloud("minta1_talaj.ply", pc_ground)
o3d.io.write_point_cloud("minta1_nemtalaj.ply", pc_non_ground)
print(f"{len(pc.points)} pont közül {len(pc_ground.points)} talajpont, {len(pc_non_ground.points)} nem talajpont.")

Talajpontok

In [ ]:
pc_show(pc_ground)

Nem talajpontok

In [ ]:
pc_show(pc_non_ground)

### Digitális domborzatmodell létrehozása

A talajpontok felhasználásával létrehozhatunk egy négyzetrács alapú digitális domborzatmodelt. Amint láthattuk a talajpontok között az alacsony növényzet és szegélyek is megjelennek.

In [ ]:
# befoglaló téglalap a talajpontokra
pnts = np.asarray(pc_ground.points)
mi = (pnts.min(axis=0)[0:2] + 0.5).astype(int)    # méterre kerekített min xy
ma = (pnts.max(axis=0)[0:2] + 0.5).astype(int)    # méterre kerekített max xy
x = pnts[:, 0]
y = pnts[:, 1]
z = pnts[:, 2]

# méteres felbontású rács létrehozása
grid_x, grid_y = np.meshgrid(
    np.linspace(mi[0], ma[0], ma[0] - mi[0]),
    np.linspace(mi[1], ma[1], ma[1] - mi[1])
)

# magasság interpolálás a rácspontokba
grid_z = griddata((x, y), z, (grid_x, grid_y), method='linear') # scipy interpoláció

# megjelenítés
plt.imshow(grid_z, extent=(x.min(), x.max(), y.min(), y.max()), origin='lower')
plt.colorbar(label='Height (z)')
plt.title("Interpolált felület")
plt.show()

### Normalizált digitális felszinmodell

A DTM és a nem talaj pontokat tartalmazó ponfelhőből levezethetünk egy normalizált felületmodellt (nDSM), ha a terepmagasságot levonjuk minden pontból. Ez abban az esetben lehet jó, ha relatív magasságokkal szeretnénk dolgozni. Például az alacsony növényzet elkülönítésére. A talaj pontokat a későbbiek miatt kihagyjuk az nDSM-ből.

![nDSM](https://www.researchgate.net/profile/Sven-Sickert/publication/341720268/figure/fig2/AS:896338581549057@1590715179150/The-Digital-Surface-Model-DSM-represents-earths-surface-and-includes-all-objects-on_W640.jpg)

Forrás: https://www.researchgate.net/profile/Sven-Sickert/publication/341720268/figure/fig2/AS:896338581549057@1590715179150/The-Digital-Surface-Model-DSM-represents-earths-surface-and-includes-all-objects-on_W640.jpg

In [ ]:
# tengelyek
x_axis = grid_x[0, :]
y_axis = grid_y[:, 0]
# interpolator létrehozása
interp = RegularGridInterpolator((y_axis, x_axis),   # FIGYLEM (y, x) sorrend
    grid_z, bounds_error=False, fill_value=np.nan)
# magasság interpoláció az összes pontra a terepen
pnts = np.asarray(pc_non_ground.points)
xy = pnts[:, :2]
# az interpolator (y, x) sorrendet vár
query_points = np.column_stack([xy[:, 1], xy[:, 0]])
# terepmagasságok
terrain_z = interp(query_points)
# normalizált magasságok
z = pnts[:, 2]
normalized_z = z - terrain_z
# normalizált pontfelhő létrehozása, színek megőrzésével
npc = o3d.geometry.PointCloud()
npts = np.column_stack((xy, normalized_z))
npc.points = o3d.utility.Vector3dVector(npts)
npc.colors = o3d.utility.Vector3dVector(pc_non_ground.colors)
o3d.io.write_point_cloud("minta1.ndsm.ply", npc)

A normalizált pontfelhő.

In [ ]:
pc_show(npc)

### Geometriai alakzatok keresése a pontfelhőben

A pontfelhő pontjainak osztályozása az adatállomány strukturáltabbá teszi, de a felhasználóknak legtöbbször a pontfelhőben található objektumok rekonstrukciójára van szüksége, melyek kevés paraméterrel leírhatók. Például ez egy oszlop (pozíció, átmérő, magasság), vagy ez egy épület (a határoló síklapok leírása). Egy kicsit hasonlót láthattunk már a digitális domborzatmodell létrehozásánál a 836533 talajpont helyett egy 43 x 35 pontot tartalmazó ráccsal helyettesítettük.

#### RANSAC módszer

A RANdom SAmple Concensus (RANSAC) módszert zajos adatokban szabályos alakzatok paramétereinek meghatározására dolgozták ki. A 3D rekonstrukciós eljárásokban széles körben alkalmazzák. A módszer alapgondolata:

1.   Válasszunk ki véletlenszerűen egy minimális ponthalmazt a modell illesztéséhez (pl. sík esetén minimum 3 nem egy egyenesre eső pontot).
2.   Végezzük el a modellillesztést a kiválasztott pontokra (pl. írjuk fel a három ponton átmenő sík egyenletét).
3.   Számoljuk meg, hogy az összes pont közül hány illeszkedik a modellünkhöz, ezeket nevezzük angolul inlier-eknek (pl. a síktól való távolsága a pontoknak kisebb mint egy korlát).
4.   Ha az eddigi legjobb modellnél több pont illeszkedik az új modellhez, akkor tekintsük ezt az eddigi legjobb modellnek.
5.   Ismételjük meg a műveleteket az 1. pontól, amíg el nem érünk egy maximális ismétlési számot.

A RANSAC módszer nem determinisztikus, ugyanazokra a pontokra kis mértékben eltérő eredményt adhat, az ismétlési szám növelése nem jár feltétlenül jobban illszkedő modell megtalálásával (nem konvergál). Az eljárás nagy előnye, hogy akkor is megbízható eredmény kaphatunk, ha az adatok nagyon zajosak.



![RANSAC folyamat](https://github.com/OSGeoLabBp/tutorials/blob/master/hungarian/ransac/ransac_line_5_48.gif?raw=true)

2D vonal illesztés példa

##### Síkok keresése

A gyakorlati feladatokban nagyon ritkán fordul elő, hogy egyetlen modellt akarunk illeszteni a teljes pontfelhőre. Például, ha a minta állományban látható templom falsíkjait szeretnénk megkapni. Ilyenkor a többszörös RANSAC sík illesztés jöhet szóba. Keressünk meg az első legtöbb pontra illeszkedő síkot, majd távolítsuk el az illeszkedő pontokat a pontfelhőből és folytassuk a sík keresést, amíg kellő számú pontra illeszkedő síkot találunk. A megoldás során a nem talaj pontok állományából indulunk ki. A síkokat a normálisuk vízszintessel bezárt szöge alapján osztályozzuk fal, tető, lapostető kategóriákba.

Induljunk ki a normalizált felszínmodellből, melyből vágjuk le a 0.5 alatti részeket (alacsony növényzet).

In [ ]:
def label(ang):
    """ return roof/wall/ground label from angle of normal to horizontal """
    label = '???'
    if abs(ang) < 15:
        label = 'fal'
    elif 80 < abs(ang) < 100:
        label = 'lapostető'
    elif 35 < abs(ang) < 55:
        label = 'tető'
    return label

In [ ]:
tol = 0.1           # tolerancia távolság a síktól
n_points = 3        # véletlenül választott pontok száma
iterations = 600
n_orig = len(npc.points)
bbox0 = o3d.geometry.AxisAlignedBoundingBox(min_bound=(550800, 182200, 0.5), max_bound=(551000, 182500, 100))
pc_w = npc.crop(bbox0)

planes = []     # megtalált síkok listája
i = 0
while True:
    plane, inliers = pc_w.segment_plane(tol, n_points, iterations)
    angle = atan2(plane[2], hypot(plane[0], plane[1])) * 180. / pi  # normális szöge a vízzintestől
    plane_pc = pc_w.select_by_index(inliers)
    o3d.io.write_point_cloud('plane_' + str(i) + '.ply', plane_pc) # sík pontok exportálása
    planes.append([plane, plane_pc, label(angle)])
    print(f"{i+1}. sik {len(plane_pc.points)} pont {label(angle)}")
    pc_w = pc_w.select_by_index(inliers, invert=True)   # folytatás a megmaradt pontokkal
    if len(plane_pc.points) < 6000:
        break
    i += 1
print(f'{len(pc_w.points)} pont maradt {n_orig}-ből')
pc_rest = pc_w.select_by_index([], invert=True)         # megmaradt pontok megőrzése a későbbiekhez

Síkok megjelenítése

In [ ]:
cols = {'fal': 'yellow', 'tető': 'red', 'lapostető': 'blue', 'egyéb': 'green'}
fig = go.Figure()
for plane in planes:
    pc_w = plane[1]
    pts = np.asarray(pc_w.voxel_down_sample(voxel_size=0.1).points)
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                               mode='markers', marker=dict(size=1, color=cols[plane[2]])))
fig.update_layout(width=800, height=600,
        scene=dict(aspectmode='data', xaxis=dict(visible=False),
                   yaxis=dict(visible=False), zaxis=dict(visible=False)),
        margin=dict(l=0, r=0, b=0, t=0),
        showlegend=False)
fig.show()

Nagyobb pontfelhők esetén a síkok vagy más alakzatok keresése a teljes pontfelhőben nem hatékony. Célszerű ilyenkor kisebb részekre felbontott részekre korlátozni a RANSAC szűrést.

##### Függőleges hengerek keresése

A kör keresztmetszetű oszlopok (függőleges tengelyű hengerek) keresése leegyszerűsíthető a vízszintes körök keresésére az XY skba vetített pontfelhőben. Itt is a RANSAC módszert alkalmazhatjuk. A feladat egyszerűsítése érdekében induljunk ki a síkok megkeresése után maradt nem talaj pontokból készített normalizált felszínmodellből, annak is a 2 és 3 méteres magasságba eső szeletéből, hogy kiszűrjük az alacsony növényzetet és a vezetékeket. A megtalált henger sugarát korlátozzuk 0.06 és 0.15 m közé. További korlátozásként a függőleges hengerpalástra eső pontok száma legyen nagyobb min 30 és magassági értelemben töltse ki a kivágat 90%-át. Mivel a sugárra, az illeszkedő pontok számára és a magassági kiterjedésre vonatkozó feltételt csak a RANSAC keresés után tudjuk figyelembe venni, várható, hogy sok nagy sugarú kört találunk, ha a teljes állományt egyben vizsgáljuk. Ennek elkerülése érdekében kisebb egységekre (voxelekre) végezzük el a feladatot.

In [ ]:
MINZ = 2
MAXZ = 3
bbox1 = o3d.geometry.AxisAlignedBoundingBox(min_bound=(550800, 182200, MINZ), max_bound=(551000, 182500, MAXZ))
col_pc = pc_rest.crop(bbox1)
pc_show(col_pc)

In [ ]:
TOLERANCE = 0.03        # tolerancia a RANSAC szűréshez
MAX_RADIUS = 0.15       # korlát a sugárra
MIN_RADIUS = 0.06
MIN = 30                # szükséges pontok száma
SIZE = 1                # voxelek vízszintes kiterjedése
columns = []
pnts = np.asarray(col_pc.points)
minp = pnts.min(axis=0).astype(np.uint32)
maxp = pnts.max(axis=0).astype(np.uint32)
for x in range(minp[0], maxp[0], SIZE):
    for y in range(minp[1], maxp[1], SIZE):
        voxel_box = o3d.geometry.AxisAlignedBoundingBox(min_bound=(x, y, MINZ), max_bound=(x+1, y+1, MAXZ))
        voxel_pnts = np.asarray(npc.crop(voxel_box).points) #[:,:2]

        while voxel_pnts.shape[0] > 10:
            cr = CircleReg(voxel_pnts)
            r = Ransac(cr)
            pfit, iterations = r.ransac_filter(tolerance=TOLERANCE)
            if pfit is None or len(pfit) < 10:
                break
            circlep = cr.get_pnts_by_index(pfit)
            cr1 = CircleReg(circlep)
            params = cr1.lkn_reg()
            if MIN_RADIUS < params[2] < MAX_RADIUS and circlep.shape[0] > MIN and\
                (np.max(circlep[:,2]) - np.min(circlep[:,2])) > 0.9 * (MAXZ -MINZ):
                columns.append(params)
                print(f"{params[0]:.2f} {params[1]:.2f} {params[2]:.2f} feltételezett oszlop, {circlep.shape[0]} pont")
            voxel_pnts = cr.get_pnts_by_index(np.logical_not(pfit))

In [ ]:
ll = []
for c in columns:
    ll.append(np.array([[c[0], c[1], 0], [c[0], c[1], 5]]))
pc_show(npc, extra=ll)

Az oszlop mellett találtunk egy lefolyót, de a másik oldali lefolyót nem találtuk meg. Megpróbálhatunk újabb feltételeket megadni, mely egyre bonyolultabbá teszi a megoldást és más helyzetekben nem biztos, hogy működik. Ilyenkor jobb eredményt hozhat a mély tanulás illetve a neurális hálózatok.